# Cross-Validation Variance (Part 2c/3) — BanglaBERT Seed [99]

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Environment:** Kaggle GPU (T4 ×2 or P100), Internet On
**Estimated runtime:** ~2 hours (1 seed × 5 folds × ~24 min/fold)

---

Part 2c of 3 — BanglaBERT Large 1×5-fold CV with seed `[99]`. Hyperparameters copied verbatim from NB1. Writes an intermediate CSV after every seed so partial results survive a session timeout.

See the **Kaggle Setup** cell below for required inputs and configuration.

## Kaggle Setup

Kaggle resets accelerator / internet / input settings after a notebook
re-import. Use this checklist before each run.

| Setting | Required value |
|---------|----------------|
| Accelerator | **GPU T4 ×2** (or P100) — required |
| Internet | **On** — required for HuggingFace model download |
| Inputs | (1) `Swarabyanjan_BEST_BALANCED_1to1.csv` (or `Swarabyanjan_Gold_Balanced_766.csv`) gold-standard dataset (766 articles); (2) HuggingFace model `csebuetnlp/banglabert_large` (downloaded via `from_pretrained` with Internet On) |

**Add as Kaggle input:**
1. Right panel → **Add Input** → **Dataset** → search for the gold-standard
   dataset (e.g., `v18-human-gold-final` or `swarabyanjan`).
2. The HuggingFace model is fetched on first run via
   `AutoTokenizer.from_pretrained('csebuetnlp/banglabert_large')` —
   Internet must be ON for the first run. (For subsequent runs you can
   pre-download the model and add it as a Kaggle dataset, then point
   `MODEL_NAME` at the local path.)

**Outputs (written to `/kaggle/working/`):**
- `cv_variance_banglabert_b3_results.json` — canonical batch output (seed 99)
- `cv_variance_banglabert_b3_results_table.csv` — flat results table
- `cv_variance_banglabert_b3_intermediate.csv` — crash-safe per-seed intermediate results

**Runtime safety.** The notebook writes `cv_variance_banglabert_b3_intermediate.csv`
after every seed (1 row maximum). If the session times out, the completed
seed will still be on disk; you can manually combine it in NB15c.

**Next steps.** After this notebook finishes, run NB15c to aggregate all
three batches (NB15b1 + NB15b2 + NB15b3) into the final 5-seed summary.


### 1. Environment Setup

In [ ]:
# === 1. Environment Setup ===
import os, sys, time, json, warnings, glob, re, math, unicodedata, random, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # headless backend (Kaggle commit mode)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, matthews_corrcoef,
                             confusion_matrix, classification_report)

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Use single GPU — prevents DataParallel OOM
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Hardware check ----
if not torch.cuda.is_available():
    raise RuntimeError(
        '\n========================================================\n'
        'NO GPU DETECTED. This notebook (NB15b3) requires a GPU.\n'
        '\n'
        'On Kaggle: right panel → Accelerator → "GPU T4 x2" or "P100".\n'
        'Then re-run this notebook top-to-bottom.\n'
        '\n'
        'If you do not want to run the BanglaBERT ablation, skip this\n'
        'notebook entirely. The aggregator (NB15c) will fall back to\n'
        'the single-seed F1=0.883 from NB1 for any missing batch.\n'
        '========================================================'
    )

DEVICE = torch.device('cuda')
print(f"PyTorch {torch.__version__}", flush=True)
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB", flush=True)
print(f"Transformers imported.", flush=True)
print(f"NB15b3: BanglaBERT Large 1x5-fold CV (GPU, ~2 hours) — seed [99]", flush=True)


### 2. Configuration

In [ ]:
# === 2. Configuration ===
SEEDS = [99]
N_FOLDS = 5

# BanglaBERT hyperparameters — copied EXACTLY from NB1_BanglaBERT_Classical.ipynb (cell 7).
MODEL_NAME = 'csebuetnlp/banglabert_large'
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# Reference single-seed F1 from NB1 (seed=42, 5-fold CV).
SINGLE_SEED_BANGLABERT_F1 = 0.8831

# Batch metadata — used by NB15c to combine all 3 batches
BATCH_ID = '2c'
TOTAL_SEEDS_PLANNED = 5
ALL_PLANNED_SEEDS = [42, 123, 2024, 7, 99]
PREVIOUS_SEEDS = [42, 123, 2024, 7]   # handled by NB15b1 + NB15b2
REMAINING_SEEDS = []                   # this is the final batch

# Output paths
RESULTS_FILE         = '/kaggle/working/cv_variance_banglabert_b3_results.json'
OUTPUT_JSON          = OUTPUT_DIR / 'cv_variance_banglabert_b3_results.json'
OUTPUT_CSV           = OUTPUT_DIR / 'cv_variance_banglabert_b3_results_table.csv'
OUTPUT_INTERMEDIATE  = OUTPUT_DIR / 'cv_variance_banglabert_b3_intermediate.csv'

print(f"BATCH_ID = {BATCH_ID}  (Part 2c/3 — final batch)")
print(f"SEEDS = {SEEDS}")
print(f"N_FOLDS = {N_FOLDS}")
print(f"Total CV fits this batch: {len(SEEDS) * N_FOLDS} (1 seed x 5 folds)")
print(f"Total seeds planned (across all 3 batches): {TOTAL_SEEDS_PLANNED}")
print(f"Previous batches' seeds: {PREVIOUS_SEEDS}")
print(f"Remaining seeds after this batch: {REMAINING_SEEDS}  (none — this is the final batch)")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LEARNING_RATE}")
print(f"WARMUP_RATIO={WARMUP_RATIO}, WEIGHT_DECAY={WEIGHT_DECAY}, fp16=True")
print(f"Reference single-seed (seed=42) F1 from NB1: {SINGLE_SEED_BANGLABERT_F1:.4f}")
print(f"RESULTS_FILE = {RESULTS_FILE}")
print(f"Intermediate CSV: {OUTPUT_INTERMEDIATE}  (crash-safe — written after each seed)")

# Helper for timestamps
def _now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


### 3. Load Gold Standard

In [ ]:
# === 3. Load Gold Standard (766 articles) ===
GOLD_FILENAME = 'Swarabyanjan_BEST_BALANCED_1to1.csv'
GOLD_FILENAME_CLEANED = 'Swarabyanjan_Gold_Balanced_766.csv'

def find_gold():
    """Search Kaggle input dirs + local paths for the gold-standard CSV."""
    for fname in [GOLD_FILENAME_CLEANED, GOLD_FILENAME]:
        candidates = [
            f'/kaggle/input/v18-human-gold-final/{fname}',
            f'/kaggle/input/swarabyanjan/{fname}',
            f'/kaggle/input/{fname}',
        ]
        for c in candidates:
            if os.path.isfile(c):
                return c
        matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if matches:
            return matches[0]
        for local in [f'./{fname}', f'../data/{fname}', f'./data/{fname}',
                      f'/home/z/my-project/analysis/github_repo/data/{fname}',
                      f'/home/z/my-project/upload/{fname}']:
            if os.path.isfile(local):
                return local
    return GOLD_FILENAME  # let pd.read_csv raise a helpful error

GOLD_PATH = find_gold()
print(f'Gold CSV path: {GOLD_PATH}')

gold = pd.read_csv(GOLD_PATH)
print(f'Gold shape: {gold.shape}')

# Normalize column names
if 'corpus_batch' not in gold.columns and 'news_source' in gold.columns:
    gold['corpus_batch'] = gold['news_source']
elif 'news_source' not in gold.columns and 'corpus_batch' in gold.columns:
    gold['news_source'] = gold['corpus_batch']

gold['headline'] = gold['headline'].fillna('').astype(str)
gold['body_text'] = gold['body_text'].fillna('').astype(str)
gold.loc[gold['body_text'] == 'not_available', 'body_text'] = ''
gold['text'] = gold['headline'] + ' ' + gold['body_text']

print(f'\nLabel distribution: {gold["best_label"].value_counts().to_dict()}')
print(f'Total: {len(gold)} | Yellow: {gold["best_label"].sum()} '
      f'| Non-yellow: {(gold["best_label"] == 0).sum()}')


### 4. BanglaBERT 1×5-Fold CV

In [ ]:
# === 4. BanglaBERT 1×5-Fold CV (1 seed × 5 folds = 5 fits, ~2 hours) ===

print('=' * 70)
print(f'[{_now()}] BanglaBERT 1x5-Fold CV  (GPU detected — running batch 2c)')
print('  Expected runtime: ~2 hours (1 seed × 5 folds × ~24 min/fold on T4)')
print('=' * 70)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(f'[{_now()}] Tokenizer loaded: {MODEL_NAME}  (vocab={len(tokenizer)})')

X_bert = gold['text'].values
y_bert = gold['best_label'].values

class BengaliNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
        )
        return {
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels': int(self.labels[idx]),
        }

def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, zero_division=0),
    }

banglabert_per_seed_results = {}  # seed -> {'per_fold_f1': [...], 'run_mean_f1': ...}
intermediate_rows = []            # written to CSV after each seed

for seed_i, seed in enumerate(SEEDS):
    print(f'\n[{_now()}] --- BanglaBERT Seed {seed} ({seed_i+1}/{len(SEEDS)}) ---')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    per_fold_f1 = []
    seed_start = time.time()
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_bert, y_bert)):
        fold_start = time.time()
        print(f'\n  [{_now()}] Seed {seed}  Fold {fold_i+1}/{N_FOLDS}  '
              f'(train={len(train_idx)}, val={len(val_idx)})')

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2
        ).to(DEVICE)

        train_ds = BengaliNewsDataset(X_bert[train_idx], y_bert[train_idx], tokenizer)
        val_ds = BengaliNewsDataset(X_bert[val_idx], y_bert[val_idx], tokenizer)
        collator = DataCollatorWithPadding(tokenizer=tokenizer)

        fold_output_dir = OUTPUT_DIR / f'nb15b3_banglabert_seed{seed}_fold{fold_i+1}'
        fold_output_dir.mkdir(parents=True, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=str(fold_output_dir),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            greater_is_better=True,
            save_total_limit=1,
            report_to='none',
            seed=seed,
            fp16=True,
            gradient_accumulation_steps=1,
            dataloader_num_workers=2,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=collator,
            compute_metrics=compute_metrics_trainer,
        )

        trainer.train()
        predictions = trainer.predict(val_ds)
        logits = predictions.predictions
        fold_preds = np.argmax(logits, axis=-1)
        fold_f1 = f1_score(y_bert[val_idx], fold_preds)
        per_fold_f1.append(float(fold_f1))
        fold_elapsed = time.time() - fold_start
        print(f'    [{_now()}] Fold {fold_i+1} F1 = {fold_f1:.4f}  ({fold_elapsed/60:.1f} min)')

        # Cleanup before next fold
        del model, trainer
        torch.cuda.empty_cache()
        if fold_output_dir.exists():
            shutil.rmtree(fold_output_dir)

    run_mean = float(np.mean(per_fold_f1))
    run_std = float(np.std(per_fold_f1))
    banglabert_per_seed_results[seed] = {
        'per_fold_f1': per_fold_f1,
        'run_mean_f1': run_mean,
        'run_std_f1': run_std,
    }
    seed_elapsed = time.time() - seed_start
    print(f'\n  [{_now()}] -> Seed {seed} mean F1 = {run_mean:.4f} ± {run_std:.4f}  ({seed_elapsed/60:.1f} min)')

    # Crash-safe intermediate save after each seed
    intermediate_rows.append({
        'seed': seed,
        'per_fold_f1': json.dumps(per_fold_f1),
        'run_mean_f1': run_mean,
        'run_std_f1': run_std,
        'seed_minutes': round(seed_elapsed / 60, 2),
        'timestamp': _now(),
    })
    pd.DataFrame(intermediate_rows).to_csv(OUTPUT_INTERMEDIATE, index=False)
    print(f'  [{_now()}] Intermediate saved: {OUTPUT_INTERMEDIATE.name} ({len(intermediate_rows)} seed(s) done)')

print(f'\n[{_now()}] BanglaBERT 1x5-fold CV (batch 2c) complete.')
print(f'  Completed seeds this batch: {list(banglabert_per_seed_results.keys())}')


### 5. Results Table

In [ ]:
# === 5. Results Table ===
rows = []
per_seed_f1_list = [banglabert_per_seed_results[s]['run_mean_f1'] for s in SEEDS
                    if s in banglabert_per_seed_results]
mean = float(np.mean(per_seed_f1_list)) if per_seed_f1_list else 0.0
std = float(np.std(per_seed_f1_list)) if per_seed_f1_list else 0.0
rows.append({
    'Model': 'BanglaBERT',
    'Batch': BATCH_ID,
    **{f'Seed {s} F1': (round(banglabert_per_seed_results[s]['run_mean_f1'], 4)
                        if s in banglabert_per_seed_results else 'n/a') for s in SEEDS},
    'Mean ± Std': f'{mean:.4f} ± {std:.4f}',
})

results_df = pd.DataFrame(rows)
print('=' * 100)
print(f'[{_now()}] CV VARIANCE RESULTS (BATCH 2c) — BanglaBERT Large  (1x5-fold CV, seed 99)')
print('=' * 100)
print(results_df.to_string(index=False))

results_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV.name}')
print(f'\nReference single-seed (seed=42) F1 from NB1: {SINGLE_SEED_BANGLABERT_F1:.4f}')
print(f'Batch 2c across-seed mean ± std:             {mean:.4f} ± {std:.4f}')


### 6. Save Results

In [ ]:
# === 6. Save Results (JSON) ===

# Build canonical batch JSON.
# NOTE: per_seed_f1 and per_seed_per_fold_f1 are keyed by str(seed) so that
# NB15c can merge the three batch outputs by seed without index alignment
# ambiguity.
per_seed_f1_dict = {str(s): banglabert_per_seed_results[s]['run_mean_f1']
                    for s in SEEDS if s in banglabert_per_seed_results}
per_seed_per_fold_f1_dict = {str(s): banglabert_per_seed_results[s]['per_fold_f1']
                             for s in SEEDS if s in banglabert_per_seed_results}
per_seed_detail = {str(s): banglabert_per_seed_results[s]
                   for s in SEEDS if s in banglabert_per_seed_results}

completed_seeds = [s for s in SEEDS if s in banglabert_per_seed_results]
per_seed_f1_list = [banglabert_per_seed_results[s]['run_mean_f1'] for s in completed_seeds]

results_json = {
    'model': 'BanglaBERT Large',
    'batch': BATCH_ID,
    'n_seeds_in_batch': len(SEEDS),
    'seeds': SEEDS,
    'completed_seeds': completed_seeds,
    'n_folds_per_seed': N_FOLDS,
    'total_seeds_planned': TOTAL_SEEDS_PLANNED,
    'all_planned_seeds': ALL_PLANNED_SEEDS,
    'previous_seeds': PREVIOUS_SEEDS,
    'remaining_seeds': REMAINING_SEEDS,
    'per_seed_f1': per_seed_f1_dict,
    'per_seed_per_fold_f1': per_seed_per_fold_f1_dict,
    'per_seed_detail': per_seed_detail,
    'batch_mean': float(np.mean(per_seed_f1_list)) if per_seed_f1_list else None,
    'batch_std': float(np.std(per_seed_f1_list)) if per_seed_f1_list else None,
    'single_seed_reference_f1': SINGLE_SEED_BANGLABERT_F1,
    'hyperparameters': {
        'model_name': MODEL_NAME,
        'max_len': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'warmup_ratio': WARMUP_RATIO,
        'weight_decay': WEIGHT_DECAY,
        'fp16': True,
    },
    'note': ('Part 2c/3: BanglaBERT seed 99. '
             'Run NB15b1 for seeds 42, 123. '
             'Run NB15b2 for seeds 2024, 7.'),
    'created_by': 'NB15b3_CV_Variance_BanglaBERT_seed_99.ipynb',
    'results_file': RESULTS_FILE,
    'completed_at': _now(),
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(results_json, f, indent=2, default=str)

print(f'Saved: {OUTPUT_JSON.name}')
print(f'\nBatch 2c final summary:')
print(f'  Completed seeds this batch: {len(completed_seeds)}/{len(SEEDS)}')
if per_seed_f1_list:
    print(f'  Per-seed F1: {per_seed_f1_list}')
    print(f'  Batch mean ± std: {results_json["batch_mean"]:.4f} ± {results_json["batch_std"]:.4f}')
    print(f'  Single-seed (NB1, seed=42) reference: {SINGLE_SEED_BANGLABERT_F1:.4f}')
print(f'\nNext steps:')
print(f'  • Run NB15c_CV_Variance_Aggregate.ipynb (CPU, ~5 min) to combine the three')
print(f'    batch outputs (NB15b1 + NB15b2 + NB15b3) with NB15a\'s SMI + classical results.')
print(f'\nIf the session timed out before the seed finished:')
print(f'  • The intermediate CSV at {OUTPUT_INTERMEDIATE.name} has the completed folds.')
print(f'  • You can manually combine them in NB15c by editing the load cell to read')
print(f'    the intermediate CSV instead of the batch JSON.')
